In [2]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [10]:
from pathlib import Path
import pandas as pd
import numpy as np

root = Path("../data/raw/car_data/car_data/train/")
stats = []
for cls_dir in sorted(root.iterdir()):
    if cls_dir.is_dir():
        n = len(list(cls_dir.glob("*.jpg"))) + len(list(cls_dir.glob("*.png")))
        stats.append({"class": cls_dir.name, "count": n})

df = pd.DataFrame(stats).sort_values("count", ascending=False)
print(df.to_string(index=False))
print(f"\nSum: {df['count'].sum()} images: {len(df)} classes")

                                                 class  count
                                   GMC Savana Van 2012     68
                               Chrysler 300 SRT-8 2010     49
              Mercedes-Benz 300-Class Convertible 1993     48
                          Mitsubishi Lancer Sedan 2012     48
                           Chevrolet Corvette ZR1 2012     47
                                    Jaguar XK XKR 2012     47
                                    Ford GT Coupe 2006     46
                            Eagle Talon Hatchback 1998     46
                                Dodge Durango SUV 2007     46
                        Volkswagen Golf Hatchback 1991     46
                               Nissan 240SX Coupe 1998     46
                                  Volvo 240 Sedan 1993     46
                             Suzuki Kizashi Sedan 2012     46
                     Bentley Continental GT Coupe 2007     46
                                    Audi S6 Sedan 2011     46
        

In [ ]:
from roboflow import Roboflow
from .env import My_ROBO_API_KEY
rf = Roboflow(api_key=My_ROBO_API_KEY)
project = rf.workspace("YOUR_WORKSPACE").project("car-brand-xx")

print("Downloading to", project.location)

In [1]:
from icrawler.builtin import GoogleImageCrawler
from pathlib import Path

# Các nhãn xe cần thu thập
brands = ["Toyota Camry car", "Honda Civic car", "BMW 3 Series car"]

for brand in brands:
    save_dir = Path(f"data/raw/google/{brand.split()[0]}_{brand.split()[1]}")
    save_dir.mkdir(parents=True, exist_ok=True)

    crawler = GoogleImageCrawler(storage={"root_dir": str(save_dir)})
    crawler.crawl(keyword=brand, max_num=100)
    n = len(list(save_dir.glob("*.jpg")))
    print(f"✅ {brand}: {n} ảnh tải về")

2026-04-08 11:42:18,755 - INFO - icrawler.crawler - start crawling...
2026-04-08 11:42:18,757 - INFO - icrawler.crawler - starting 1 feeder threads...
2026-04-08 11:42:18,766 - INFO - feeder - thread feeder-001 exit
2026-04-08 11:42:18,769 - INFO - icrawler.crawler - starting 1 parser threads...
2026-04-08 11:42:18,787 - INFO - icrawler.crawler - starting 1 downloader threads...
2026-04-08 11:42:19,028 - INFO - parser - parsing result page https://www.google.com/search?q=Toyota+Camry+car&ijn=0&start=0&tbs=&tbm=isch
Exception in thread parser-001:
Traceback (most recent call last):
  File "d:\anaconda\envs\ojt-ai\lib\threading.py", line 1016, in _bootstrap_inner
    self.run()
  File "d:\anaconda\envs\ojt-ai\lib\threading.py", line 953, in run
    self._target(*self._args, **self._kwargs)
  File "d:\anaconda\envs\ojt-ai\lib\site-packages\icrawler\parser.py", line 93, in worker_exec
    for task in self.parse(response, **kwargs):
TypeError: 'NoneType' object is not iterable
2026-04-08 11

✅ Toyota Camry car: 0 ảnh tải về


Exception in thread parser-001:
Traceback (most recent call last):
  File "d:\anaconda\envs\ojt-ai\lib\threading.py", line 1016, in _bootstrap_inner
    self.run()
  File "d:\anaconda\envs\ojt-ai\lib\threading.py", line 953, in run
    self._target(*self._args, **self._kwargs)
  File "d:\anaconda\envs\ojt-ai\lib\site-packages\icrawler\parser.py", line 93, in worker_exec
    for task in self.parse(response, **kwargs):
TypeError: 'NoneType' object is not iterable
2026-04-08 11:42:29,836 - INFO - downloader - no more download task for thread downloader-001
2026-04-08 11:42:29,840 - INFO - downloader - thread downloader-001 exit
2026-04-08 11:42:29,885 - INFO - icrawler.crawler - Crawling task done!
2026-04-08 11:42:29,893 - INFO - icrawler.crawler - start crawling...
2026-04-08 11:42:29,894 - INFO - icrawler.crawler - starting 1 feeder threads...
2026-04-08 11:42:29,895 - INFO - feeder - thread feeder-001 exit
2026-04-08 11:42:29,898 - INFO - icrawler.crawler - starting 1 parser threads..

✅ Honda Civic car: 0 ảnh tải về


2026-04-08 11:42:30,431 - INFO - parser - parsing result page https://www.google.com/search?q=BMW+3+Series+car&ijn=0&start=0&tbs=&tbm=isch
Exception in thread parser-001:
Traceback (most recent call last):
  File "d:\anaconda\envs\ojt-ai\lib\threading.py", line 1016, in _bootstrap_inner
    self.run()
  File "d:\anaconda\envs\ojt-ai\lib\threading.py", line 953, in run
    self._target(*self._args, **self._kwargs)
  File "d:\anaconda\envs\ojt-ai\lib\site-packages\icrawler\parser.py", line 93, in worker_exec
    for task in self.parse(response, **kwargs):
TypeError: 'NoneType' object is not iterable
2026-04-08 11:42:34,910 - INFO - downloader - no more download task for thread downloader-001
2026-04-08 11:42:34,912 - INFO - downloader - thread downloader-001 exit
2026-04-08 11:42:35,914 - INFO - icrawler.crawler - Crawling task done!


✅ BMW 3 Series car: 0 ảnh tải về


# Bài 2: Làm sạch và tổ chức dữ liệu

In [ ]:
import shutil, random
from pathlib import Path
from collections import defaultdict

def split_data(
    src_dir: str,
    dst_dir: str,
    train_ratio=0.7,
    val_ratio=0.15,
    seed=42
):
    random.seed(seed)
    src = Path(src_dir)
    dst = Path(dst_dir)
    
    for split in ["train", "val", "test"]:
        (dst/split).mkdir(parents=True, exist_ok=True)
        
    for cls_dir in sorted(src.iterdir()):
        if not cls_dir.is_dir():
            continue
        images = list(cls_dir.glob("*.jpg")) + list(cls_dir.glob("*.png"))
        random.shuffle(images)
        
        n = len(images)
        n_train = int(n * train_ratio)
        n_val = int(n * val_ratio)
        
        splits = {
            "train": images[:n_train],
            "val": images[n_train:n_train + n_val],
            "test": images[n_train + n_val:],
        }
        
        for split, files in splits.items():
            out_dir = dst / split / cls_dir.name
            out_dir.mkdir(parents=True, exist_ok=True)
            for file in files:
                shutil.copy(file, out_dir / file.name)
        print(f"{cls_dir.name:30s}: train={len(splits['train'])} val={len(splits['val'])} test={len(splits['test'])}")

split_data("../data/raw/car_data/car_data/train", "../data/processed/car_brand")
print("\n✅ Dataset split complete!")

FileNotFoundError: [WinError 3] The system cannot find the path specified: 'data\\raw\\car_data\\car_data\\train'